In [1]:
%pip install --upgrade pip -U langchain langchain-openai langchain langchain-anthropic langchain-core langchain-graph-retriever langchain-community langchain-experimental langchain-text-splitters langchain-neo4j langchain-ollama neo4j python-dotenv yfiles_jupyter_graphs

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached langchain-1.3.11-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_openai-1.3.3-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_anthropic-1.4.8-py3-none-any.whl.metadata (3.5 kB)
  Using cached langchain_core-1.4.8-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_graph_retriever-0.8.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_neo4j-0.10.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached neo4j-6.2.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached yfiles_jupyter_graphs-1.10.11-py3-none-any

In [2]:
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field
from neo4j import GraphDatabase, Driver

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector, GraphCypherQAChain
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_experimental.graph_transformers import LLMGraphTransformer
from yfiles_jupyter_graphs import GraphWidget
from langchain_core.documents import Document
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI


load_dotenv()

/tmp/ipykernel_7408/1968259090.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/tmp/ipykernel_7408/1968259090.py:18: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


True

In [3]:
graph = Neo4jGraph(database="cloud3")

In [29]:
import os

chunk_id = 1

directory = "../data/chapters/"
for file in os.listdir(directory):
    loader = TextLoader(file_path=f"../data/chapters/{file}")
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=200)
    documents = text_splitter.split_documents(documents=docs)

    with open(f"../data/chunks/chunked_{file}", "w") as file:
        file.write(str(documents))


In [1]:
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship


def serialize_entitiy(entity):
    return Node(id=entity["id"], type=entity["type"], properties=entity["properties"])


def serialize_relationship(relation, nodes: list[Node]):
    source = next(node for node in nodes if node.id == relation["source"])
    target = next(node for node in nodes if node.id == relation["target"])
    return Relationship(
        source=source,
        target=target,
        type=relation["type"],
        properties=relation["properties"],
    )

In [19]:
import requests
print(requests.get("https://chat-ai.academiccloud.de/v1/models", timeout=5, auth=f"Bearer {os.getenv("OPEN_AI_API_KEY")}", json=True).json())

TypeError: 'str' object is not callable

In [24]:
cloud_llm = ChatOpenAI(
    model="gemma-4-31b-it",
    base_url="https://chat-ai.academiccloud.de/v1/",
    api_key=os.getenv("OPEN_AI_API_KEY")
)

cloud_llm.invoke("Say hello in one Sentence!")

AIMessage(content='Hello there!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 19, 'total_tokens': 23, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma-4-31b-it', 'system_fingerprint': 'vllm-0.23.0-tp2-7c9232a3', 'id': 'chatcmpl-a4e0d54903d432cd', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0508-78a0-7f51-92e8-ba06ce1dd15c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 4, 'total_tokens': 23, 'input_token_details': {}, 'output_token_details': {}})

In [ ]:
from copy import deepcopy
import os
import json


# Merge Nodes
all_nodes = []
all_relationships = []
next_free_id = 1
id_map = {}


directory = "../data/extraction_results/"
for file in os.listdir(directory):

    with open(
        f"../data/extraction_results/{file}", "r", encoding="utf-8"
    ) as f:
        results = json.load(f)


    for node in results["nodes"]:
        curr_node = next(
            (
                n
                for n in all_nodes
                if n["type"] == node["type"]
                and n["properties"]["description"] == node["properties"]["description"]
            ),
            None,
        )

        if curr_node is not None:
            curr_props = curr_node["properties"]
            new_props = node["properties"]
            new_keys = new_props.keys()

            for key in new_keys:
                if key in curr_props:
                    if isinstance(new_props[key], str):
                        if curr_props[key] != new_props[key]:
                            attributes = curr_props[key].split(',') + new_props[key].split(',')
                            curr_props[key] = ', '.join(dict.fromkeys(attributes))
                            # curr_props[key] = ' '.join(dict.fromkeys((', '.join([curr_props[key], new_props[key]])).split()))
                            
                        pass
                    else:
                        curr_props[key] = list(set(curr_props[key] + new_props[key]))
                else:
                    curr_props[key] = new_props[key]
            
            id_map[node["id"]] = curr_node


            
        else:
            canonical_node = deepcopy(node)
            canonical_node["id"] = next_free_id
            all_nodes.append(canonical_node)
            id_map[node["id"]] = canonical_node
            next_free_id+=1

    for relationship in results["relationships"]:
        
        canonical_source_node = id_map.get(relationship["source"])
        canonical_target_node = id_map.get(relationship["target"])
        curr_rel = next(
            (
                r
                for r in all_relationships
                if r["type"] == relationship["type"]
                and r["source"] == canonical_source_node["id"] and r["target"] == canonical_target_node["id"]
            ),
            None,
        )

        if curr_rel is not None:
            curr_props = curr_rel["properties"]
            new_props = relationship["properties"]
            new_keys = new_props.keys()

            for key in new_keys:
                if key in curr_props:
                    if isinstance(new_props[key], str):
                        if curr_props[key] != new_props[key]:
                            attributes = curr_props[key].split(',') + new_props[key].split(',')
                            curr_props[key] = ', '.join(dict.fromkeys(attributes))
                            # curr_props[key] = ' '.join(dict.fromkeys((', '.join([curr_props[key], new_props[key]])).split()))
                            
                        pass
                    else:
                        curr_props[key] += new_props[key]
                else:
                    curr_props[key] = new_props[key]

        else:
            canonical_relationship = deepcopy(relationship)
            canonical_relationship["source"] = canonical_source_node["id"]
            canonical_relationship["target"] = canonical_target_node["id"]
            all_relationships.append(canonical_relationship)
    
            
print(f"nodes: {len(all_nodes)}")
print(f"relationships: {len(all_relationships)}")    

nodes: 319
relationships: 305


In [79]:
with open("../data/merged_results/relationships.json", "w", encoding="utf-8") as f:
    json.dump(all_relationships, f, indent=4)

In [ ]:
print(f"Serializing nodes...")
node_objects: list[Node] = []
for node in all_nodes:
    node_objects.append(serialize_entitiy(node))

print(f"Serializing relationships...")
relationship_objects: list[Relationship] = []
for rel in all_relationships:
    relationship_objects.append(serialize_relationship(rel, node_objects))

# print(node_objects)
print(f"Uploading results...")
graph.add_graph_documents(
    [GraphDocument(nodes=node_objects, relationships=relationship_objects)],
    baseEntityLabel=False,
    include_source=False,
)

Serializing nodes...
Serializing relationships...
Uploading results...


In [64]:
dump = [
    GraphDocument(
        nodes=[],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Sherlock Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Sherlock Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='IS_ADMIRER_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men\'s motives and actions. But for the trained'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='and a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men\'s motives and actions. But for the trained\nreasoner to admit such intrusions into his own delicate and finely\nadjusted temperament was to introduce a distracting factor which might\nthrow a doubt upon all his mental results. Grit in a sensitive\ninstrument, or a crack in one of his own high-power lenses, would not\nbe more disturbing than a strong emotion in a nature such as his. And\nyet there was but one woman to him, and that woman was the late Irene\nAdler, of dubious and questionable memory.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={}),
            Node(id='official police', type='Organisation', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='RESIDES_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='official police', type='Organisation', properties={}),
                type='ASSISTS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='I had seen little of Holmes lately. My marriage had drifted us away\nfrom each other. My own complete happiness, and the home-centred\ninterests which rise up around the man who first finds himself master\nof his own establishment, were sufficient to absorb all my attention,\nwhile Holmes, who loathed every form of society with his whole Bohemian\nsoul, remained in our lodgings in Baker Street, buried among his old\nbooks, and alternating from week to week between cocaine and ambition,\nthe drowsiness of the drug, and the fierce energy of his own keen\nnature. He was still, as ever, deeply attracted by the study of crime,\nand occupied his immense faculties and extraordinary powers of'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Odessa', type='Location', properties={}),
            Node(id='Trepoff murder', type='Case', properties={}),
            Node(id='Atkinson brothers', type='Person', properties={}),
            Node(id='Trincomalee', type='Location', properties={}),
            Node(id='reigning family of Holland', type='Organisation', properties={}),
            Node(id='official police', type='Organisation', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Odessa', type='Location', properties={}),
                type='SUMMONED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Trepoff murder', type='Case', properties={}),
                type='INVESTIGATED',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Trincomalee', type='Location', properties={}),
                type='INVESTIGATED_CASE_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='reigning family of Holland', type='Organisation', properties={}),
                type='COMPLETED_MISSION_FOR',
                properties={}
            ),
            Relationship(
                source=Node(id='official police', type='Organisation', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='ABANDONED_CASES_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='the drowsiness of the drug, and the fierce energy of his own keen\nnature. He was still, as ever, deeply attracted by the study of crime,\nand occupied his immense faculties and extraordinary powers of\nobservation in following out those clues, and clearing up those\nmysteries which had been abandoned as hopeless by the official police.\nFrom time to time I heard some vague account of his doings: of his\nsummons to Odessa in the case of the Trepoff murder, of his clearing up\nof the singular tragedy of the Atkinson brothers at Trincomalee, and\nfinally of the mission which he had accomplished so delicately and\nsuccessfully for the reigning family of Holland. Beyond these signs of'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='reigning family of Holland', type='Organisation', properties={}),
            Node(id='Holland', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holland', type='Location', properties={}),
                target=Node(id='reigning family of Holland', type='Organisation', properties={}),
                type='GOVERNED_BY',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='finally of the mission which he had accomplished so delicately and\nsuccessfully for the reigning family of Holland. Beyond these signs of\nhis activity, however, which I merely shared with all the readers of\nthe daily press, I knew little of my former friend and companion.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Study in Scarlet', type='Case', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='PASSED_THROUGH',
                properties={}
            ),
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='VISITED',
                properties={}
            ),
            Relationship(
                source=Node(id='Baker Street', type='Location', properties={}),
                target=Node(id='Study in Scarlet', type='Case', properties={}),
                type='ASSOCIATED_WITH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='One night—it was on the twentieth of March, 1888—I was returning from a\njourney to a patient (for I had now returned to civil practice), when\nmy way led me through Baker Street. As I passed the well-remembered\ndoor, which must always be associated in my mind with my wooing, and\nwith the dark incidents of the Study in Scarlet, I was seized with a\nkeen desire to see Holmes again, and to know how he was employing his\nextraordinary powers. His rooms were brilliantly lit, and, even as I\nlooked up, I saw his tall, spare figure pass twice in a dark silhouette\nagainst the blind. He was pacing the room swiftly, eagerly, with his\nhead sunk upon his chest and his hands clasped behind him. To me, who'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='against the blind. He was pacing the room swiftly, eagerly, with his\nhead sunk upon his chest and his hands clasped behind him. To me, who\nknew his every mood and habit, his attitude and manner told their own\nstory. He was at work again. He had risen out of his drug-created\ndreams and was hot upon the scent of some new problem. I rang the bell\nand was shown up to the chamber which had formerly been in part my own.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Mary Jane', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='DEDUCES_ABOUT',
                properties={}
            ),
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Mary Jane', type='Person', properties={}),
                type='HAS_SERVANT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"His manner was not effusive. It seldom was; but he was glad, I think,\nto see me. With hardly a word spoken, but with a kindly eye, he waved\nme to an armchair, threw across his case of cigars, and indicated a\nspirit case and a gasogene in the corner. Then he stood before the fire\nand looked me over in his singular introspective fashion.\n\n"Wedlock suits you," he remarked. "I think, Watson, that you have put\non seven and a half pounds since I saw you."\n\n"Seven!" I answered.\n\n"Indeed, I should have thought a little more. Just a trifle more, I\nfancy, Watson. And in practice again, I observe. You did not tell me\nthat you intended to go into harness."\n\n"Then, how do you know?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Mary Jane', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Mary Jane', type='Person', properties={}),
                type='EMPLOYS',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='DEDUCES_ABOUT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Then, how do you know?"\n\n"I see it, I deduce it. How do I know that you have been getting\nyourself very wet lately, and that you have a most clumsy and careless\nservant girl?"\n\n"My dear Holmes," said I, "this is too much. You would certainly have\nbeen burned, had you lived a few centuries ago. It is true that I had a\ncountry walk on Thursday and came home in a dreadful mess, but as I\nhave changed my clothes I can\'t imagine how you deduce it. As to Mary\nJane, she is incorrigible, and my wife has given her notice, but there,\nagain, I fail to see how you work it out."\n\nHe chuckled to himself and rubbed his long, nervous hands together.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='London', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='London', type='Location', properties={}),
                type='OPERATES_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"It is simplicity itself," said he; "my eyes tell me that on the inside\nof your left shoe, just where the firelight strikes it, the leather is\nscored by six almost parallel cuts. Obviously they have been caused by\nsomeone who has very carelessly scraped round the edges of the sole in\norder to remove crusted mud from it. Hence, you see, my double\ndeduction that you had been out in vile weather, and that you had a\nparticularly malignant boot-slitting specimen of the London slavey. As\nto your practice, if a gentleman walks into my rooms smelling of\niodoform, with a black mark of nitrate of silver upon his right\nforefinger, and a bulge on the right side of his top-hat to show where'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='DEDUCES_PROFESSION_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='to your practice, if a gentleman walks into my rooms smelling of\niodoform, with a black mark of nitrate of silver upon his right\nforefinger, and a bulge on the right side of his top-hat to show where\nhe has secreted his stethoscope, I must be dull, indeed, if I do not\npronounce him to be an active member of the medical profession."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='CHRONICLES_CASES_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='I could not help laughing at the ease with which he explained his\nprocess of deduction. "When I hear you give your reasons," I remarked,\n"the thing always appears to me to be so ridiculously simple that I\ncould easily do it myself, though at each successive instance of your\nreasoning I am baffled until you explain your process. And yet I\nbelieve that my eyes are as good as yours."\n\n"Quite so," he answered, lighting a cigarette, and throwing himself\ndown into an armchair. "You see, but you do not observe. The\ndistinction is clear. For example, you have frequently seen the steps\nwhich lead up from the hall to this room."\n\n"Frequently."\n\n"How often?"\n\n"Well, some hundreds of times."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='RESIDES_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Frequently."\n\n"How often?"\n\n"Well, some hundreds of times."\n\n"Then how many are there?"\n\n"How many?... I don\'t know."\n\n"Quite so! You have not observed. And yet you have seen. That is just\nmy point. Now, I know that there are seventeen steps, because I have\nboth seen and observed. By the way, since you are interested in these\nlittle problems, and since you are good enough to chronicle one or two\nof my trifling experiences, you may be interested in this." He threw\nover a sheet of thick, pink-tinted notepaper which had been lying open\nupon the table. "It came by the last post," said he. "Read it aloud."\n\nThe note was undated, and without either signature or address.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Europe', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Europe', type='Location', properties={}),
                type='SERVED_ROYAL_HOUSES_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='The note was undated, and without either signature or address.\n\n"There will call upon you to-night, at a quarter to eight o\'clock," it\nsaid, "a gentleman who desires to consult you upon a matter of the very\ndeepest moment. Your recent services to one of the royal houses of\nEurope have shown that you are one who may safely be trusted with\nmatters which are of an importance which can hardly be exaggerated.\nThis account of you we have from all quarters received. Be in your\nchamber then at that hour, and do not take it amiss if your visitor\nwear a mask."\n\n"This is indeed a mystery," I remarked. "What do you imagine that it\nmeans?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='CONSULTS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"This is indeed a mystery," I remarked. "What do you imagine that it\nmeans?"\n\n"I have no data yet. It is a capital mistake to theorise before one has\ndata. Insensibly one begins to twist facts to suit theories, instead of\ntheories to suit facts. But the note itself. What do you deduce from\nit?"\n\nI carefully examined the writing, and the paper upon which it was\nwritten.\n\n"The man who wrote it was presumably well to do," I remarked,\nendeavouring to imitate my companion\'s processes. "Such paper could not\nbe bought under half a crown a packet. It is peculiarly strong and\nstiff."\n\n"Peculiar—that is the very word," said Holmes. "It is not an English\npaper at all. Hold it up to the light."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Peculiar—that is the very word," said Holmes. "It is not an English\npaper at all. Hold it up to the light."\n\nI did so, and saw a large "E" with a small "g," a "P," and a large "G"\nwith a small "t" woven into the texture of the paper.\n\n"What do you make of that?" asked Holmes.\n\n"The name of the maker, no doubt; or his monogram, rather."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Egria', type='Location', properties={}),
            Node(id='Bohemia', type='Location', properties={}),
            Node(id='Carlsbad', type='Location', properties={}),
            Node(id='Wallenstein', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Egria', type='Location', properties={}),
                target=Node(id='Bohemia', type='Location', properties={}),
                type='LOCATED_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='Egria', type='Location', properties={}),
                target=Node(id='Carlsbad', type='Location', properties={}),
                type='NEAR',
                properties={}
            ),
            Relationship(
                source=Node(id='Wallenstein', type='Person', properties={}),
                target=Node(id='Egria', type='Location', properties={}),
                type='DIED_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Not at all. The \'G\' with the small \'t\' stands for \'Gesellschaft,\'\nwhich is the German for \'Company.\' It is a customary contraction like\nour \'Co.\' \'P,\' of course, stands for \'Papier.\' Now for the \'Eg.\' Let us\nglance at our Continental Gazetteer." He took down a heavy brown volume\nfrom his shelves. "Eglow, Eglonitz—here we are, Egria. It is in a\nGerman-speaking country—in Bohemia, not far from Carlsbad. \'Remarkable\nas being the scene of the death of Wallenstein, and for its numerous\nglass-factories and paper-mills.\' Ha, ha, my boy, what do you make of\nthat?" His eyes sparkled, and he sent up a great blue triumphant cloud\nfrom his cigarette.\n\n"The paper was made in Bohemia," I said.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Bohemia', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Bohemia', type='Location', properties={}),
                type='DEDUCED_ORIGIN_FROM',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"The paper was made in Bohemia," I said.\n\n"Precisely. And the man who wrote the note is a German. Do you note the\npeculiar construction of the sentence—\'This account of you we have from\nall quarters received.\' A Frenchman or Russian could not have written\nthat. It is the German who is so uncourteous to his verbs. It only\nremains, therefore, to discover what is wanted by this German who\nwrites upon Bohemian paper and prefers wearing a mask to showing his\nface. And here he comes, if I am not mistaken, to resolve all our\ndoubts."\n\nAs he spoke there was the sharp sound of horses\' hoofs and grating\nwheels against the curb, followed by a sharp pull at the bell. Holmes\nwhistled.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='WORKS_WITH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='As he spoke there was the sharp sound of horses\' hoofs and grating\nwheels against the curb, followed by a sharp pull at the bell. Holmes\nwhistled.\n\n"A pair, by the sound," said he. "Yes," he continued, glancing out of\nthe window. "A nice little brougham and a pair of beauties. A hundred\nand fifty guineas apiece. There\'s money in this case, Watson, if there\nis nothing else."\n\n"I think that I had better go, Holmes."\n\n"Not a bit, Doctor. Stay where you are. I am lost without my Boswell.\nAnd this promises to be interesting. It would be a pity to miss it."\n\n"But your client—"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='WORKS_WITH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Not a bit, Doctor. Stay where you are. I am lost without my Boswell.\nAnd this promises to be interesting. It would be a pity to miss it."\n\n"But your client—"\n\n"Never mind him. I may want your help, and so may he. Here he comes.\nSit down in that armchair, Doctor, and give us your best attention."\n\nA slow and heavy step, which had been heard upon the stairs and in the\npassage, paused immediately outside the door. Then there was a loud and\nauthoritative tap.\n\n"Come in!" said Holmes.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='A man entered who could hardly have been less than six feet six inches\nin height, with the chest and limbs of a Hercules. His dress was rich\nwith a richness which would, in England, be looked upon as akin to bad\ntaste. Heavy bands of astrakhan were slashed across the sleeves and\nfronts of his double-breasted coat, while the deep blue cloak which was\nthrown over his shoulders was lined with flame-coloured silk and\nsecured at the neck with a brooch which consisted of a single flaming\nberyl. Boots which extended halfway up his calves, and which were\ntrimmed at the tops with rich brown fur, completed the impression of\nbarbaric opulence which was suggested by his whole appearance. He'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='trimmed at the tops with rich brown fur, completed the impression of\nbarbaric opulence which was suggested by his whole appearance. He\ncarried a broad-brimmed hat in his hand, while he wore across the upper\npart of his face, extending down past the cheekbones, a black vizard\nmask, which he had apparently adjusted that very moment, for his hand\nwas still raised to it as he entered. From the lower part of the face\nhe appeared to be a man of strong character, with a thick, hanging lip,\nand a long, straight chin suggestive of resolution pushed to the length\nof obstinacy.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Count Von Kramm', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Bohemia', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Count Von Kramm', type='Person', properties={}),
                target=Node(id='Bohemia', type='Location', properties={}),
                type='NOBLEMAN_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Count Von Kramm', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='CONSULTS',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='INTRODUCES',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"You had my note?" he asked with a deep harsh voice and a strongly\nmarked German accent. "I told you that I would call." He looked from\none to the other of us, as if uncertain which to address.\n\n"Pray take a seat," said Holmes. "This is my friend and colleague, Dr.\nWatson, who is occasionally good enough to help me in my cases. Whom\nhave I the honour to address?"\n\n"You may address me as the Count Von Kramm, a Bohemian nobleman. I\nunderstand that this gentleman, your friend, is a man of honour and\ndiscretion, whom I may trust with a matter of the most extreme\nimportance. If not, I should much prefer to communicate with you\nalone."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Count Von Kramm', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Count Von Kramm', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='BINDS_TO_SECRECY',
                properties={}
            ),
            Relationship(
                source=Node(id='Count Von Kramm', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='BINDS_TO_SECRECY',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='I rose to go, but Holmes caught me by the wrist and pushed me back into\nmy chair. "It is both, or none," said he. "You may say before this\ngentleman anything which you may say to me."\n\nThe Count shrugged his broad shoulders. "Then I must begin," said he,\n"by binding you both to absolute secrecy for two years; at the end of\nthat time the matter will be of no importance.... At present it is not too\nmuch to say that it is of such weight it may have an influence upon\nEuropean history."\n\n"I promise," said Holmes.\n\n"And I."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='House of Ormstein', type='Organisation', properties={}),
            Node(id='Bohemia', type='Location', properties={}),
            Node(id='Count Von Kramm', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='House of Ormstein', type='Organisation', properties={}),
                target=Node(id='Bohemia', type='Location', properties={}),
                type='HEREDITARY_KINGS_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Count Von Kramm', type='Person', properties={}),
                target=Node(id='House of Ormstein', type='Organisation', properties={}),
                type='REPRESENTS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"I promise," said Holmes.\n\n"And I."\n\n"You will excuse this mask," continued our strange visitor. "The august\nperson who employs me wishes his agent to be unknown to you, and I may\nconfess at once that the title by which I have just called myself is\nnot exactly my own."\n\n"I was aware of it," said Holmes dryly.\n\n"The circumstances are of great delicacy, and every precaution has to\nbe taken to quench what might grow to be an immense scandal and\nseriously compromise one of the reigning families of Europe. To speak\nplainly, the matter implicates the great House of Ormstein, hereditary\nkings of Bohemia."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Wilhelm Gottsreich Sigismond von Ormstein', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Wilhelm Gottsreich Sigismond von Ormstein', type='Person', properties={}),
                type='IS_SAME_AS',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='IDENTIFIES',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"If your Majesty would condescend to state your case," he remarked, "I\nshould be better able to advise you."\n\nThe man sprang from his chair and paced up and down the room in\nuncontrollable agitation. Then, with a gesture of desperation, he tore\nthe mask from his face and hurled it upon the ground. "You are right,"\nhe cried; "I am the King. Why should I attempt to conceal it?"\n\n"Why, indeed?" murmured Holmes. "Your Majesty had not spoken before I\nwas aware that I was addressing Wilhelm Gottsreich Sigismond von\nOrmstein, Grand Duke of Cassel-Felstein, and hereditary King of\nBohemia."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Warsaw', type='Location', properties={}),
            Node(id='Prague', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='MET_IN',
                properties={'location': 'Warsaw'}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Prague', type='Location', properties={}),
                type='TRAVELLED_FROM',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='CONSULTS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"But you can understand," said our strange visitor, sitting down once\nmore and passing his hand over his high white forehead, "you can\nunderstand that I am not accustomed to doing such business in my own\nperson. Yet the matter was so delicate that I could not confide it to\nan agent without putting myself in his power. I have come _incognito_\nfrom Prague for the purpose of consulting you."\n\n"Then, pray consult," said Holmes, shutting his eyes once more.\n\n"The facts are briefly these: Some five years ago, during a lengthy\nvisit to Warsaw, I made the acquaintance of the well-known adventuress,\nIrene Adler. The name is no doubt familiar to you."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='New Jersey', type='Location', properties={}),
            Node(id='La Scala', type='Organisation', properties={}),
            Node(id='Imperial Opera of Warsaw', type='Organisation', properties={}),
            Node(id='London', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='New Jersey', type='Location', properties={}),
                type='BORN_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='La Scala', type='Organisation', properties={}),
                type='PERFORMED_AT',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Imperial Opera of Warsaw', type='Organisation', properties={}),
                type='WAS_PRIMA_DONNA_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='London', type='Location', properties={}),
                type='LIVES_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Let me see!" said Holmes. "Hum! Born in New Jersey in the year 1858.\nContralto—hum! La Scala, hum! Prima donna Imperial Opera of Warsaw—yes!\nRetired from operatic stage—ha! Living in London—quite so! Your\nMajesty, as I understand, became entangled with this young person,\nwrote her some compromising letters, and is now desirous of getting\nthose letters back."\n\n"Precisely so. But how—"\n\n"Was there a secret marriage?"\n\n"None."\n\n"No legal papers or certificates?"\n\n"None."\n\n"Then I fail to follow your Majesty. If this young person should\nproduce her letters for blackmailing or other purposes, how is she to\nprove their authenticity?"\n\n"There is the writing."\n\n"Pooh, pooh! Forgery."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='PHOTOGRAPHED_WITH',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='POSSESSES_COMPROMISING_PHOTOGRAPH_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"There is the writing."\n\n"Pooh, pooh! Forgery."\n\n"My private note-paper."\n\n"Stolen."\n\n"My own seal."\n\n"Imitated."\n\n"My photograph."\n\n"Bought."\n\n"We were both in the photograph."\n\n"Oh, dear! That is very bad! Your Majesty has indeed committed an\nindiscretion."\n\n"I was mad—insane."\n\n"You have compromised yourself seriously."\n\n"I was only Crown Prince then. I was young. I am but thirty now."\n\n"It must be recovered."\n\n"We have tried and failed."\n\n"Your Majesty must pay. It must be bought."\n\n"She will not sell."\n\n"Stolen, then."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='HIRES',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='BLACKMAILS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"It must be recovered."\n\n"We have tried and failed."\n\n"Your Majesty must pay. It must be bought."\n\n"She will not sell."\n\n"Stolen, then."\n\n"Five attempts have been made. Twice burglars in my pay ransacked her\nhouse. Once we diverted her luggage when she travelled. Twice she has\nbeen waylaid. There has been no result."\n\n"No sign of it?"\n\n"Absolutely none."\n\nHolmes laughed. "It is quite a pretty little problem," said he.\n\n"But a very serious one to me," returned the King reproachfully.\n\n"Very, indeed. And what does she propose to do with the photograph?"\n\n"To ruin me."\n\n"But how?"\n\n"I am about to be married."\n\n"So I have heard."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Clotilde Lothman von Saxe-Meningen', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Scandinavia', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Clotilde Lothman von Saxe-Meningen', type='Person', properties={}),
                type='BETROTHED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Clotilde Lothman von Saxe-Meningen', type='Person', properties={}),
                target=Node(id='Scandinavia', type='Location', properties={}),
                type='DAUGHTER_OF_KING_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='THREATENS_TO_EXPOSE',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Very, indeed. And what does she propose to do with the photograph?"\n\n"To ruin me."\n\n"But how?"\n\n"I am about to be married."\n\n"So I have heard."\n\n"To Clotilde Lothman von Saxe-Meningen, second daughter of the King of\nScandinavia. You may know the strict principles of her family. She is\nherself the very soul of delicacy. A shadow of a doubt as to my conduct\nwould bring the matter to an end."\n\n"And Irene Adler?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='THREATENS_WITH_PHOTOGRAPH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"And Irene Adler?"\n\n"Threatens to send them the photograph. And she will do it. I know that\nshe will do it. You do not know her, but she has a soul of steel. She\nhas the face of the most beautiful of women, and the mind of the most\nresolute of men. Rather than I should marry another woman, there are no\nlengths to which she would not go—none."\n\n"You are sure that she has not sent it yet?"\n\n"I am sure."\n\n"And why?"\n\n"Because she has said that she would send it on the day when the\nbetrothal was publicly proclaimed. That will be next Monday."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Langham Hotel', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Langham Hotel', type='Location', properties={}),
                type='STAYING_AT',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='UPDATES',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"You are sure that she has not sent it yet?"\n\n"I am sure."\n\n"And why?"\n\n"Because she has said that she would send it on the day when the\nbetrothal was publicly proclaimed. That will be next Monday."\n\n"Oh, then we have three days yet," said Holmes with a yawn. "That is\nvery fortunate, as I have one or two matters of importance to look into\njust at present. Your Majesty will, of course, stay in London for the\npresent?"\n\n"Certainly. You will find me at the Langham under the name of the Count\nVon Kramm."\n\n"Then I shall drop you a line to let you know how we progress."\n\n"Pray do so. I shall be all anxiety."\n\n"Then, as to money?"... \n\n"You have _carte blanche_."\n\n"Absolutely?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={}),
            Node(id='Serpentine Avenue', type='Location', properties={}),
            Node(id="St. John's Wood", type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='RESIDES_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='Briony Lodge', type='Location', properties={}),
                target=Node(id='Serpentine Avenue', type='Location', properties={}),
                type='LOCATED_ON',
                properties={}
            ),
            Relationship(
                source=Node(id='Serpentine Avenue', type='Location', properties={}),
                target=Node(id="St. John's Wood", type='Location', properties={}),
                type='LOCATED_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='PAYS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='Holmes scribbled a receipt upon a sheet of his note-book and handed it\nto him.\n\n"And Mademoiselle\'s address?" he asked.\n\n"Is Briony Lodge, Serpentine Avenue, St. John\'s Wood."\n\nHolmes took a note of it. "One other question," said he. "Was the\nphotograph a cabinet?"\n\n"It was."\n\n"Then, good-night, your Majesty, and I trust that we shall soon have\nsome good news for you. And good-night, Watson," he added, as the\nwheels of the royal brougham rolled down the street. "If you will be\ngood enough to call to-morrow afternoon at three o\'clock I should like\nto chat this little matter over with you."\n\n\nII.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='VISITED',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='COLLABORATES_WITH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='At three o\'clock precisely I was at Baker Street, but Holmes had not\nyet returned. The landlady informed me that he had left the house\nshortly after eight o\'clock in the morning. I sat down beside the fire,\nhowever, with the intention of awaiting him, however long he might be.\nI was already deeply interested in his inquiry, for, though it was\nsurrounded by none of the grim and strange features which were\nassociated with the two crimes which I have already recorded, still,\nthe nature of the case and the exalted station of his client gave it a\ncharacter of its own. Indeed, apart from the nature of the\ninvestigation which my friend had on hand, there was something in his'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='the nature of the case and the exalted station of his client gave it a\ncharacter of its own. Indeed, apart from the nature of the\ninvestigation which my friend had on hand, there was something in his\nmasterly grasp of a situation, and his keen, incisive reasoning, which\nmade it a pleasure to me to study his system of work, and to follow the\nquick, subtle methods by which he disentangled the most inextricable\nmysteries. So accustomed was I to his invariable success that the very\npossibility of his failing had ceased to enter into my head.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='RETURNED_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='It was close upon four before the door opened, and a drunken-looking\ngroom, ill-kempt and side-whiskered, with an inflamed face and\ndisreputable clothes, walked into the room. Accustomed as I was to my\nfriend\'s amazing powers in the use of disguises, I had to look three\ntimes before I was certain that it was indeed he. With a nod he\nvanished into the bedroom, whence he emerged in five minutes\ntweed-suited and respectable, as of old. Putting his hands into his\npockets, he stretched out his legs in front of the fire and laughed\nheartily for some minutes.\n\n"Well, really!" he cried, and then he choked and laughed again until he\nwas obliged to lie back, limp and helpless, in the chair.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='SURVEILLED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Well, really!" he cried, and then he choked and laughed again until he\nwas obliged to lie back, limp and helpless, in the chair.\n\n"What is it?"\n\n"It\'s quite too funny. I am sure you could never guess how I employed\nmy morning, or what I ended by doing."\n\n"I can\'t imagine. I suppose that you have been watching the habits, and\nperhaps the house, of Miss Irene Adler."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='SURVEILLED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Quite so; but the sequel was rather unusual. I will tell you, however.\nI left the house a little after eight o\'clock this morning in the\ncharacter of a groom out of work. There is a wonderful sympathy and\nfreemasonry among horsey men. Be one of them, and you will know all\nthat there is to know. I soon found Briony Lodge. It is a _bijou_\nvilla, with a garden at the back, but built out in front right up to\nthe road, two stories. Chubb lock to the door. Large sitting-room on\nthe right side, well furnished, with long windows almost to the floor,\nand those preposterous English window fasteners which a child could\nopen. Behind there was nothing remarkable, save that the passage window'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='INSPECTED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='and those preposterous English window fasteners which a child could\nopen. Behind there was nothing remarkable, save that the passage window\ncould be reached from the top of the coach-house. I walked round it and\nexamined it closely from every point of view, but without noting\nanything else of interest.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='GATHERED_INTELLIGENCE_ON',
                properties={}
            ),
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='FREQUENT_VISITOR_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"I then lounged down the street and found, as I expected, that there\nwas a mews in a lane which runs down by one wall of the garden. I lent\nthe ostlers a hand in rubbing down their horses, and received in\nexchange twopence, a glass of half-and-half, two fills of shag tobacco,\nand as much information as I could desire about Miss Adler, to say\nnothing of half a dozen other people in the neighbourhood in whom I was\nnot in the least interested, but whose biographies I was compelled to\nlisten to."\n\n"And what of Irene Adler?" I asked.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Inner Temple', type='Organisation', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Inner Temple', type='Organisation', properties={}),
                type='MEMBER_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='VISITS_REGULARLY',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Oh, she has turned all the men\'s heads down in that part. She is the\ndaintiest thing under a bonnet on this planet. So say the\nSerpentine-mews, to a man. She lives quietly, sings at concerts, drives\nout at five every day, and returns at seven sharp for dinner. Seldom\ngoes out at other times, except when she sings. Has only one male\nvisitor, but a good deal of him. He is dark, handsome, and dashing,\nnever calls less than once a day, and often twice. He is a Mr. Godfrey\nNorton, of the Inner Temple. See the advantages of a cabman as a\nconfidant. They had driven him home a dozen times from Serpentine-mews,\nand knew all about him. When I had listened to all they had to tell, I'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='OBSERVED',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='INVESTIGATED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='confidant. They had driven him home a dozen times from Serpentine-mews,\nand knew all about him. When I had listened to all they had to tell, I\nbegan to walk up and down near Briony Lodge once more, and to think\nover my plan of campaign.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Inner Temple', type='Organisation', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Inner Temple', type='Organisation', properties={}),
                type='HAS_CHAMBERS_IN',
                properties={}
            ),
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='LEGAL_ASSOCIATE_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"This Godfrey Norton was evidently an important factor in the matter.\nHe was a lawyer. That sounded ominous. What was the relation between\nthem, and what the object of his repeated visits? Was she his client,\nhis friend, or his mistress? If the former, she had probably\ntransferred the photograph to his keeping. If the latter, it was less\nlikely. On the issue of this question depended whether I should\ncontinue my work at Briony Lodge, or turn my attention to the\ngentleman\'s chambers in the Temple. It was a delicate point, and it\nwidened the field of my inquiry. I fear that I bore you with these\ndetails, but I have to let you see my little difficulties, if you are'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='REPORTS_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='widened the field of my inquiry.... I fear that I bore you with these\ndetails, but I have to let you see my little difficulties, if you are\nto understand the situation."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='ARRIVED_AT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"I am following you closely," I answered.\n\n"I was still balancing the matter in my mind when a hansom cab drove up\nto Briony Lodge, and a gentleman sprang out. He was a remarkably\nhandsome man, dark, aquiline, and moustached—evidently the man of whom\nI had heard. He appeared to be in a great hurry, shouted to the cabman\nto wait, and brushed past the maid who opened the door with the air of\na man who was thoroughly at home.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Church of St. Monica', type='Location', properties={}),
            Node(id='Regent Street', type='Location', properties={}),
            Node(id='Edgeware Road', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Church of St. Monica', type='Location', properties={}),
                type='HEADED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Church of St. Monica', type='Location', properties={}),
                target=Node(id='Edgeware Road', type='Location', properties={}),
                type='LOCATED_ON',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"He was in the house about half an hour, and I could catch glimpses of\nhim in the windows of the sitting-room, pacing up and down, talking\nexcitedly, and waving his arms. Of her I could see nothing. Presently\nhe emerged, looking even more flurried than before. As he stepped up to\nthe cab, he pulled a gold watch from his pocket and looked at it\nearnestly, \'Drive like the devil,\' he shouted, \'first to Gross &\nHankey\'s in Regent Street, and then to the Church of St. Monica in the\nEdgeware Road. Half a guinea if you do it in twenty minutes!\''
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Church of St. Monica', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Church of St. Monica', type='Location', properties={}),
                type='RUSHED_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Away they went, and I was just wondering whether I should not do well\nto follow them when up the lane came a neat little landau, the coachman\nwith his coat only half-buttoned, and his tie under his ear, while all\nthe tags of his harness were sticking out of the buckles. It hadn\'t\npulled up before she shot out of the hall door and into it. I only\ncaught a glimpse of her at the moment, but she was a lovely woman, with\na face that a man might die for.\n\n"\'The Church of St. Monica, John,\' she cried, \'and half a sovereign if\nyou reach it in twenty minutes.\''
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Church of St. Monica', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Church of St. Monica', type='Location', properties={}),
                type='FOLLOWED_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"\'The Church of St. Monica, John,\' she cried, \'and half a sovereign if\nyou reach it in twenty minutes.\'\n\n"This was quite too good to lose, Watson. I was just balancing whether\nI should run for it, or whether I should perch behind her landau when a\ncab came through the street. The driver looked twice at such a shabby\nfare, but I jumped in before he could object. \'The Church of St.\nMonica,\' said I, \'and half a sovereign if you reach it in twenty\nminutes.\' It was twenty-five minutes to twelve, and of course it was\nclear enough what was in the wind.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Church of St. Monica', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='MARRIED',
                properties={'location': 'Church of St. Monica'}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='WITNESSED_MARRIAGE_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"My cabby drove fast. I don\'t think I ever drove faster, but the others\nwere there before us. The cab and the landau with their steaming horses\nwere in front of the door when I arrived. I paid the man and hurried\ninto the church. There was not a soul there save the two whom I had\nfollowed and a surpliced clergyman, who seemed to be expostulating with\nthem. They were all three standing in a knot in front of the altar. I\nlounged up the side aisle like any other idler who has dropped into a\nchurch. Suddenly, to my surprise, the three at the altar faced round to\nme, and Godfrey Norton came running as hard as he could towards me.\n\n"\'Thank God,\' he cried. \'You\'ll do. Come! Come!\''
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='SERVED_AS_WITNESS_FOR',
                properties={}
            ),
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='MARRIED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"\'Thank God,\' he cried. \'You\'ll do. Come! Come!\'\n\n"\'What then?\' I asked.\n\n"\'Come, man, come, only three minutes, or it won\'t be legal.\''
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='MARRIED',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='ACTED_AS_WITNESS_FOR',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"I was half-dragged up to the altar, and before I knew where I was I\nfound myself mumbling responses which were whispered in my ear, and\nvouching for things of which I knew nothing, and generally assisting in\nthe secure tying up of Irene Adler, spinster, to Godfrey Norton,\nbachelor. It was all done in an instant, and there was the gentleman\nthanking me on the one side and the lady on the other, while the\nclergyman beamed on me in front. It was the most preposterous position\nin which I ever found myself in my life, and it was the thought of it\nthat started me laughing just now. It seems that there had been some\ninformality about their license, that the clergyman absolutely refused'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='REWARDED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='that started me laughing just now. It seems that there had been some\ninformality about their license, that the clergyman absolutely refused\nto marry them without a witness of some sort, and that my lucky\nappearance saved the bridegroom from having to sally out into the\nstreets in search of a best man. The bride gave me a sovereign, and I\nmean to wear it on my watch chain in memory of the occasion."'
        )
    ),
    GraphDocument(
        nodes=[
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Inner Temple', type='Organisation', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Godfrey Norton', type='Person', properties={}),
                target=Node(id='Inner Temple', type='Organisation', properties={}),
                type='RETURNED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='RETURNED_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"This is a very unexpected turn of affairs," said I; "and what then?"\n\n"Well, I found my plans very seriously menaced. It looked as if the\npair might take an immediate departure, and so necessitate very prompt\nand energetic measures on my part. At the church door, however, they\nseparated, he driving back to the Temple, and she to her own house. \'I\nshall drive out in the park at five as usual,\' she said as she left\nhim. I heard no more. They drove away in different directions, and I\nwent off to make my own arrangements."\n\n"Which are?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Mrs. Turner', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='RECRUITS_HELP_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Mrs. Turner', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='PROVIDES_FOOD_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Which are?"\n\n"Some cold beef and a glass of beer," he answered, ringing the bell. "I\nhave been too busy to think of food, and I am likely to be busier still\nthis evening. By the way, Doctor, I shall want your co-operation."\n\n"I shall be delighted."\n\n"You don\'t mind breaking the law?"\n\n"Not in the least."\n\n"Nor running a chance of arrest?"\n\n"Not in a good cause."\n\n"Oh, the cause is excellent!"\n\n"Then I am your man."\n\n"I was sure that I might rely on you."\n\n"But what is it you wish?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Mrs. Turner', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Mrs. Turner', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='SERVES_AS_LANDLADY_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='PLANS_OPERATION_AT',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='RETURNS_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Nor running a chance of arrest?"\n\n"Not in a good cause."\n\n"Oh, the cause is excellent!"\n\n"Then I am your man."\n\n"I was sure that I might rely on you."\n\n"But what is it you wish?"\n\n"When Mrs. Turner has brought in the tray I will make it clear to you.\nNow," he said as he turned hungrily on the simple fare that our\nlandlady had provided, "I must discuss it while I eat, for I have not\nmuch time. It is nearly five now. In two hours we must be on the scene\nof action. Miss Irene, or Madame, rather, returns from her drive at\nseven. We must be at Briony Lodge to meet her."\n\n"And what then?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='INSTRUCTS',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='PLANS_INFILTRATION_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"And what then?"\n\n"You must leave that to me. I have already arranged what is to occur.\nThere is only one point on which I must insist. You must not interfere,\ncome what may. You understand?"\n\n"I am to be neutral?"\n\n"To do nothing whatever. There will probably be some small\nunpleasantness. Do not join in it. It will end in my being conveyed\ninto the house. Four or five minutes afterwards the sitting-room window\nwill open. You are to station yourself close to that open window."\n\n"Yes."\n\n"You are to watch me, for I will be visible to you."\n\n"Yes."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='ASSIGNS_TASK_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Yes."\n\n"You are to watch me, for I will be visible to you."\n\n"Yes."\n\n"And when I raise my hand—so—you will throw into the room what I give\nyou to throw, and will, at the same time, raise the cry of fire. You\nquite follow me?"\n\n"Entirely."\n\n"It is nothing very formidable," he said, taking a long cigar-shaped\nroll from his pocket. "It is an ordinary plumber\'s smoke-rocket, fitted\nwith a cap at either end to make it self-lighting. Your task is\nconfined to that. When you raise your cry of fire, it will be taken up\nby quite a number of people. You may then walk to the end of the\nstreet, and I will rejoin you in ten minutes. I hope that I have made\nmyself clear?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='CONFIRMS_INSTRUCTIONS_FROM',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"I am to remain neutral, to get near the window, to watch you, and at\nthe signal to throw in this object, then to raise the cry of fire, and\nto wait you at the corner of the street."\n\n"Precisely."\n\n"Then you may entirely rely on me."\n\n"That is excellent. I think, perhaps, it is almost time that I prepare\nfor the new role I have to play."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='John Hare', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='John Hare', type='Person', properties={}),
                type='COMPARED_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"That is excellent. I think, perhaps, it is almost time that I prepare\nfor the new role I have to play."\n\nHe disappeared into his bedroom and returned in a few minutes in the\ncharacter of an amiable and simple-minded Nonconformist clergyman. His\nbroad black hat, his baggy trousers, his white tie, his sympathetic\nsmile, and general look of peering and benevolent curiosity were such\nas Mr. John Hare alone could have equalled. It was not merely that\nHolmes changed his costume. His expression, his manner, his very soul\nseemed to vary with every fresh part that he assumed. The stage lost a\nfine actor, even as science lost an acute reasoner, when he became a\nspecialist in crime.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={}),
            Node(id='Serpentine Avenue', type='Location', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='DEPARTED_FROM',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Serpentine Avenue', type='Location', properties={}),
                type='ARRIVED_AT',
                properties={}
            ),
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='STAKED_OUT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='It was a quarter past six when we left Baker Street, and it still\nwanted ten minutes to the hour when we found ourselves in Serpentine\nAvenue. It was already dusk, and the lamps were just being lighted as\nwe paced up and down in front of Briony Lodge, waiting for the coming\nof its occupant. The house was just such as I had pictured it from\nSherlock Holmes\' succinct description, but the locality appeared to be\nless private than I expected. On the contrary, for a small street in a\nquiet neighbourhood, it was remarkably animated. There was a group of\nshabbily dressed men smoking and laughing in a corner, a\nscissors-grinder with his wheel, two guardsmen who were flirting with a'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='quiet neighbourhood, it was remarkably animated. There was a group of\nshabbily dressed men smoking and laughing in a corner, a\nscissors-grinder with his wheel, two guardsmen who were flirting with a\nnurse-girl, and several well-dressed young men who were lounging up and\ndown with cigars in their mouths.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='SEARCHES_FOR_PHOTOGRAPH_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='MARRIED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='PHOTOGRAPHED_WITH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"You see," remarked Holmes, as we paced to and fro in front of the\nhouse, "this marriage rather simplifies matters. The photograph becomes\na double-edged weapon now. The chances are that she would be as averse\nto its being seen by Mr. Godfrey Norton, as our client is to its coming\nto the eyes of his princess. Now the question is, Where are we to find\nthe photograph?"\n\n"Where, indeed?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='ATTEMPTED_TO_WAYLAY',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Where, indeed?"\n\n"It is most unlikely that she carries it about with her. It is cabinet\nsize. Too large for easy concealment about a woman\'s dress. She knows\nthat the King is capable of having her waylaid and searched. Two\nattempts of the sort have already been made. We may take it, then, that\nshe does not carry it about with her."\n\n"Where, then?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='DEDUCES_HIDING_PLACE_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='STORES_PHOTOGRAPH_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Where, then?"\n\n"Her banker or her lawyer. There is that double possibility. But I am\ninclined to think neither. Women are naturally secretive, and they like\nto do their own secreting. Why should she hand it over to anyone else?\nShe could trust her own guardianship, but she could not tell what\nindirect or political influence might be brought to bear upon a\nbusiness man. Besides, remember that she had resolved to use it within\na few days. It must be where she can lay her hands upon it. It must be\nin her own house."\n\n"But it has twice been burgled."\n\n"Pshaw! They did not know how to look."\n\n"But how will you look?"\n\n"I will not look."\n\n"What then?"\n\n"I will get her to show me."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='PLANS_TO_TRICK',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"But it has twice been burgled."\n\n"Pshaw! They did not know how to look."\n\n"But how will you look?"\n\n"I will not look."\n\n"What then?"\n\n"I will get her to show me."\n\n"But she will refuse."\n\n"She will not be able to. But I hear the rumble of wheels. It is her\ncarriage. Now carry out my orders to the letter."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='ARRIVED_AT',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='STAGED_INCIDENT_NEAR',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='As he spoke the gleam of the sidelights of a carriage came round the\ncurve of the avenue. It was a smart little landau which rattled up to\nthe door of Briony Lodge. As it pulled up, one of the loafing men at\nthe corner dashed forward to open the door in the hope of earning a\ncopper, but was elbowed away by another loafer, who had rushed up with\nthe same intention. A fierce quarrel broke out, which was increased by\nthe two guardsmen, who took sides with one of the loungers, and by the\nscissors-grinder, who was equally hot upon the other side. A blow was\nstruck, and in an instant the lady, who had stepped from her carriage,\nwas the centre of a little knot of flushed and struggling men, who'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='FEIGNED_INJURY_TO_ASSIST',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='ENTERED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='struck, and in an instant the lady, who had stepped from her carriage,\nwas the centre of a little knot of flushed and struggling men, who\nstruck savagely at each other with their fists and sticks. Holmes\ndashed into the crowd to protect the lady; but, just as he reached her,\nhe gave a cry and dropped to the ground, with the blood running freely\ndown his face. At his fall the guardsmen took to their heels in one\ndirection and the loungers in the other, while a number of better\ndressed people, who had watched the scuffle without taking part in it,\ncrowded in to help the lady and to attend to the injured man. Irene\nAdler, as I will still call her, had hurried up the steps; but she'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='STOOD_AT_ENTRANCE_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='crowded in to help the lady and to attend to the injured man. Irene\nAdler, as I will still call her, had hurried up the steps; but she\nstood at the top with her superb figure outlined against the lights of\nthe hall, looking back into the street.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='INVITED_INTO_HOME',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='CARRIED_INTO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Is the poor gentleman much hurt?" she asked.\n\n"He is dead," cried several voices.\n\n"No, no, there\'s life in him!" shouted another. "But he\'ll be gone\nbefore you can get him to hospital."\n\n"He\'s a brave fellow," said a woman. "They would have had the lady\'s\npurse and watch if it hadn\'t been for him. They were a gang, and a\nrough one, too. Ah, he\'s breathing now."\n\n"He can\'t lie in the street. May we bring him in, marm?"\n\n"Surely. Bring him into the sitting-room. There is a comfortable sofa.\nThis way, please!"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='CARRIED_INTO',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='CARED_FOR',
                properties={}
            ),
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='OBSERVED_FROM_WINDOW',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='Slowly and solemnly he was borne into Briony Lodge and laid out in the\nprincipal room, while I still observed the proceedings from my post by\nthe window. The lamps had been lit, but the blinds had not been drawn,\nso that I could see Holmes as he lay upon the couch. I do not know\nwhether he was seized with compunction at that moment for the part he\nwas playing, but I know that I never felt more heartily ashamed of\nmyself in my life than when I saw the beautiful creature against whom I\nwas conspiring, or the grace and kindliness with which she waited upon\nthe injured man. And yet it would be the blackest treachery to Holmes'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='EXECUTES_PLAN_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='was conspiring, or the grace and kindliness with which she waited upon\nthe injured man. And yet it would be the blackest treachery to Holmes\nto draw back now from the part which he had intrusted to me. I hardened\nmy heart, and took the smoke-rocket from under my ulster. After all, I\nthought, we are not injuring her. We are but preventing her from\ninjuring another.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Watson', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='THREW_SMOKE_ROCKET_INTO',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='SIGNALLED_FROM_WITHIN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='Holmes had sat up upon the couch, and I saw him motion like a man who\nis in need of air. A maid rushed across and threw open the window. At\nthe same instant I saw him raise his hand and at the signal I tossed my\nrocket into the room with a cry of "Fire!" The word was no sooner out\nof my mouth than the whole crowd of spectators, well dressed and\nill—gentlemen, ostlers, and servant maids—joined in a general shriek of\n"Fire!" Thick clouds of smoke curled through the room and out at the\nopen window. I caught a glimpse of rushing figures, and a moment later\nthe voice of Holmes from within assuring them that it was a false\nalarm. Slipping through the shouting crowd I made my way to the corner'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Edgeware Road', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='REGROUPED_WITH',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Edgeware Road', type='Location', properties={}),
                type='WALKED_TOWARDS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='the voice of Holmes from within assuring them that it was a false\nalarm. Slipping through the shouting crowd I made my way to the corner\nof the street, and in ten minutes was rejoiced to find my friend\'s arm\nin mine, and to get away from the scene of uproar. He walked swiftly\nand in silence for some few minutes until we had turned down one of the\nquiet streets which lead towards the Edgeware Road.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='REVEALED_PHOTOGRAPH_LOCATION_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='DEBRIEFS',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"You did it very nicely, Doctor," he remarked. "Nothing could have been\nbetter. It is all right."\n\n"You have the photograph?"\n\n"I know where it is."\n\n"And how did you find out?"\n\n"She showed me, as I told you she would."\n\n"I am still in the dark."\n\n"I do not wish to make a mystery," said he, laughing. "The matter was\nperfectly simple. You, of course, saw that everyone in the street was\nan accomplice. They were all engaged for the evening."\n\n"I guessed as much."\n\n"Then, when the row broke out, I had a little moist red paint in the\npalm of my hand. I rushed forward, fell down, clapped my hand to my\nface, and became a piteous spectacle. It is an old trick."\n\n"That also I could fathom."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='INFILTRATED_HOME_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='STORES_PHOTOGRAPH_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"That also I could fathom."\n\n"Then they carried me in. She was bound to have me in. What else could\nshe do? And into her sitting-room, which was the very room which I\nsuspected. It lay between that and her bedroom, and I was determined to\nsee which. They laid me on a couch, I motioned for air, they were\ncompelled to open the window, and you had your chance."\n\n"How did that help you?"'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Darlington Substitution Scandal', type='Case', properties={}),
            Node(id='Arnsworth Castle', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Darlington Substitution Scandal', type='Case', properties={}),
                type='PREVIOUSLY_SOLVED',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Arnsworth Castle', type='Location', properties={}),
                type='PREVIOUSLY_INVESTIGATED',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='RUSHED_TO_RETRIEVE_PHOTOGRAPH_IN',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"It was all-important. When a woman thinks that her house is on fire,\nher instinct is at once to rush to the thing which she values most. It\nis a perfectly overpowering impulse, and I have more than once taken\nadvantage of it. In the case of the Darlington Substitution Scandal it\nwas of use to me, and also in the Arnsworth Castle business. A married\nwoman grabs at her baby; an unmarried one reaches for her jewel-box.\nNow it was clear to me that our lady of to-day had nothing in the house\nmore precious to her than what we are in quest of. She would rush to\nsecure it. The alarm of fire was admirably done. The smoke and shouting'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='HID_PHOTOGRAPH_IN',
                properties={'location_detail': 'recess behind sliding panel above right bell-pull'}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='LOCATED_PHOTOGRAPH_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='more precious to her than what we are in quest of. She would rush to\nsecure it. The alarm of fire was admirably done. The smoke and shouting\nwere enough to shake nerves of steel. She responded beautifully. The\nphotograph is in a recess behind a sliding panel just above the right\nbell-pull. She was there in an instant, and I caught a glimpse of it as\nshe half drew it out. When I cried out that it was a false alarm, she\nreplaced it, glanced at the rocket, rushed from the room, and I have\nnot seen her since. I rose, and, making my excuses, escaped from the\nhouse. I hesitated whether to attempt to secure the photograph at once;'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='ESCAPED_FROM',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='not seen her since. I rose, and, making my excuses, escaped from the\nhouse. I hesitated whether to attempt to secure the photograph at once;\nbut the coachman had come in, and as he was watching me narrowly, it\nseemed safer to wait. A little over-precipitance may ruin all."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='PLANS_TO_RETRIEVE_PHOTOGRAPH_WITH',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='PLANS_RETURN_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"And now?" I asked.\n\n"Our quest is practically finished. I shall call with the King\nto-morrow, and with you, if you care to come with us. We will be shown\ninto the sitting-room to wait for the lady, but it is probable that\nwhen she comes she may find neither us nor the photograph. It might be\na satisfaction to his Majesty to regain it with his own hands."\n\n"And when will you call?"\n\n"At eight in the morning. She will not be up, so that we shall have a\nclear field. Besides, we must be prompt, for this marriage may mean a\ncomplete change in her life and habits. I must wire to the King without\ndelay."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='RETURNED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='GREETED_IN_DISGUISE',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='We had reached Baker Street and had stopped at the door. He was\nsearching his pockets for the key when someone passing said:\n\n"Good-night, Mister Sherlock Holmes."\n\nThere were several people on the pavement at the time, but the greeting\nappeared to come from a slim youth in an ulster who had hurried by.\n\n"I\'ve heard that voice before," said Holmes, staring down the dimly lit\nstreet. "Now, I wonder who the deuce that could have been."\n\n\nIII.\n\nI slept at Baker Street that night, and we were engaged upon our toast\nand coffee in the morning when the King of Bohemia rushed into the\nroom.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Baker Street', type='Location', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Baker Street', type='Location', properties={}),
                type='RUSHED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='MARRIED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='TRAVELS_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='III.\n\nI slept at Baker Street that night, and we were engaged upon our toast\nand coffee in the morning when the King of Bohemia rushed into the\nroom.\n\n"You have really got it!" he cried, grasping Sherlock Holmes by either\nshoulder and looking eagerly into his face.\n\n"Not yet."\n\n"But you have hopes?"\n\n"I have hopes."\n\n"Then, come. I am all impatience to be gone."\n\n"We must have a cab."\n\n"No, my brougham is waiting."\n\n"Then that will simplify matters." We descended and started off once\nmore for Briony Lodge.\n\n"Irene Adler is married," remarked Holmes.\n\n"Married! When?"\n\n"Yesterday."\n\n"But to whom?"\n\n"To an English lawyer named Norton."\n\n"But she could not love him."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Serpentine Avenue', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='MARRIED_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='REGRETS_LOSING',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Serpentine Avenue', type='Location', properties={}),
                type='ARRIVED_AT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"I am in hopes that she does."\n\n"And why in hopes?"\n\n"Because it would spare your Majesty all fear of future annoyance. If\nthe lady loves her husband, she does not love your Majesty. If she does\nnot love your Majesty, there is no reason why she should interfere with\nyour Majesty\'s plan."\n\n"It is true. And yet—! Well! I wish she had been of my own station!\nWhat a queen she would have made!" He relapsed into a moody silence,\nwhich was not broken until we drew up in Serpentine Avenue.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={}),
            Node(id='Charing Cross', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Charing Cross', type='Location', properties={}),
                type='DEPARTED_FROM',
                properties={'destination': 'Continent'}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='FLED_WITH',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='ARRIVED_TOO_LATE_AT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"It is true. And yet—! Well! I wish she had been of my own station!\nWhat a queen she would have made!" He relapsed into a moody silence,\nwhich was not broken until we drew up in Serpentine Avenue.\n\nThe door of Briony Lodge was open, and an elderly woman stood upon the\nsteps. She watched us with a sardonic eye as we stepped from the\nbrougham.\n\n"Mr. Sherlock Holmes, I believe?" said she.\n\n"I am Mr. Holmes," answered my companion, looking at her with a\nquestioning and rather startled gaze.\n\n"Indeed! My mistress told me that you were likely to call. She left\nthis morning with her husband by the 5:15 train from Charing Cross for\nthe Continent."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='OUTWITTED',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='PANICS_BEFORE',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Indeed! My mistress told me that you were likely to call. She left\nthis morning with her husband by the 5:15 train from Charing Cross for\nthe Continent."\n\n"What!" Sherlock Holmes staggered back, white with chagrin and\nsurprise. "Do you mean that she has left England?"\n\n"Never to return."\n\n"And the papers?" asked the King hoarsely. "All is lost."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Briony Lodge', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Briony Lodge', type='Location', properties={}),
                type='SEARCHED',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='LEFT_LETTER_FOR',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='LEFT_PHOTOGRAPH_FOR',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"We shall see." He pushed past the servant and rushed into the\ndrawing-room, followed by the King and myself. The furniture was\nscattered about in every direction, with dismantled shelves and open\ndrawers, as if the lady had hurriedly ransacked them before her flight.\nHolmes rushed at the bell-pull, tore back a small sliding shutter, and,\nplunging in his hand, pulled out a photograph and a letter. The\nphotograph was of Irene Adler herself in evening dress, the letter was\nsuperscribed to "Sherlock Holmes, Esq. To be left till called for." My\nfriend tore it open, and we all three read it together. It was dated at\nmidnight of the preceding night and ran in this way:'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='WROTE_LETTER_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='WARNED_AGAINST_HOLMES',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"MY DEAR MR. SHERLOCK HOLMES,—You really did it very well. You took\n    me in completely. Until after the alarm of fire, I had not a\n    suspicion. But then, when I found how I had betrayed myself, I\n    began to think. I had been warned against you months ago. I had\n    been told that, if the King employed an agent, it would certainly\n    be you. And your address had been given me. Yet, with all this, you\n    made me reveal what you wanted to know. Even after I became\n    suspicious, I found it hard to think evil of such a dear, kind old\n    clergyman. But, you know, I have been trained as an actress myself.\n    Male costume is nothing new to me. I often take advantage of the'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='John the coachman', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='John the coachman', type='Person', properties={}),
                type='SENT_TO_WATCH_HOLMES',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='COUNTER_SURVEILLED',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='clergyman. But, you know, I have been trained as an actress myself.\n    Male costume is nothing new to me. I often take advantage of the\n    freedom which it gives. I sent John, the coachman, to watch you,\n    ran upstairs, got into my walking clothes, as I call them, and came\n    down just as you departed.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Inner Temple', type='Organisation', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='FOLLOWED_TO_HOME_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Inner Temple', type='Organisation', properties={}),
                type='VISITED_HUSBAND_AT',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Well, I followed you to your door, and so made sure that I was\n    really an object of interest to the celebrated Mr. Sherlock Holmes.\n    Then I, rather imprudently, wished you good-night, and started for\n    the Temple to see my husband.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Godfrey Norton', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Godfrey Norton', type='Person', properties={}),
                type='FLED_WITH',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='RENOUNCES_THREAT_AGAINST',
                properties={}
            ),
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='WROTE_FAREWELL_LETTER_TO',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"We both thought the best resource was flight, when pursued by so\n    formidable an antagonist; so you will find the nest empty when you\n    call to-morrow. As to the photograph, your client may rest in\n    peace. I love and am loved by a better man than he. The King may do\n    what he will without hindrance from one whom he has cruelly\n    wronged. I keep it only to safeguard myself, and to preserve a\n    weapon which will always secure me from any steps which he might\n    take in the future. I leave a photograph which he might care to\n    possess; and I remain, dear Mr. Sherlock Holmes,\n\n\n    "Very truly yours,\n\n    "IRENE NORTON, _née_ ADLER."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='ADMIRES',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='REBUKES',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"Very truly yours,\n\n    "IRENE NORTON, _née_ ADLER."\n\n\n"What a woman—oh, what a woman!" cried the King of Bohemia, when we had\nall three read this epistle. "Did I not tell you how quick and resolute\nshe was? Would she not have made an admirable queen? Is it not a pity\nthat she was not on my level?"\n\n"From what I have seen of the lady, she seems, indeed, to be on a very\ndifferent level to your Majesty," said Holmes coldly. "I am sorry that\nI have not been able to bring your Majesty\'s business to a more\nsuccessful conclusion."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='OFFERS_REWARD_TO',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='REQUESTS_PHOTOGRAPH_OF',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='TRUSTS_WORD_OF',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"On the contrary, my dear sir," cried the King; "nothing could be more\nsuccessful. I know that her word is inviolate. The photograph is now as\nsafe as if it were in the fire."\n\n"I am glad to hear your Majesty say so."\n\n"I am immensely indebted to you. Pray tell me in what way I can reward\nyou. This ring—" He slipped an emerald snake ring from his finger and\nheld it out upon the palm of his hand.\n\n"Your Majesty has something which I should value even more highly,"\nsaid Holmes.\n\n"You have but to name it."\n\n"This photograph!"\n\nThe King stared at him in amazement.\n\n"Irene\'s photograph!" he cried. "Certainly, if you wish it."'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='King of Bohemia', type='Person', properties={}),
                type='DISMISSES',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Watson', type='Person', properties={}),
                type='DEPARTS_WITH',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='"You have but to name it."\n\n"This photograph!"\n\nThe King stared at him in amazement.\n\n"Irene\'s photograph!" he cried. "Certainly, if you wish it."\n\n"I thank your Majesty. Then there is no more to be done in the matter.\nI have the honour to wish you a very good morning." He bowed, and,\nturning away without observing the hand which the King had stretched\nout to him, he set off in my company for his chambers.'
        )
    ),

    GraphDocument(
        nodes=[
            Node(id='Holmes', type='Person', properties={}),
            Node(id='Irene Adler', type='Person', properties={}),
            Node(id='King of Bohemia', type='Person', properties={}),
            Node(id='Watson', type='Person', properties={}),
            Node(id='Bohemia', type='Location', properties={})
        ],
        relationships=[
            Relationship(
                source=Node(id='Irene Adler', type='Person', properties={}),
                target=Node(id='Holmes', type='Person', properties={}),
                type='DEFEATED',
                properties={}
            ),
            Relationship(
                source=Node(id='Holmes', type='Person', properties={}),
                target=Node(id='Irene Adler', type='Person', properties={}),
                type='HOLDS_IN_HIGHEST_REGARD',
                properties={}
            ),
            Relationship(
                source=Node(id='King of Bohemia', type='Person', properties={}),
                target=Node(id='Bohemia', type='Location', properties={}),
                type='RULES',
                properties={}
            )
        ],
        source=Document(
            metadata={'source': '../data/chapters/chapter_1.txt'},
            page_content='And that was how a great scandal threatened to affect the kingdom of\nBohemia, and how the best plans of Mr. Sherlock Holmes were beaten by a\nwoman\'s wit. He used to make merry over the cleverness of women, but I\nhave not heard him do it of late. And when he speaks of Irene Adler, or\nwhen he refers to her photograph, it is always under the honourable\ntitle of _the_ woman.'
        )
    )
]

In [ ]:
# directory = "../data/graph_documents/"
# for doc in os.listdir(directory):

graph.add_graph_documents(
    dump,
    baseEntityLabel=True,
    include_source=True,
)

In [3]:
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
vector_retriever = vector_index.as_retriever()

In [71]:
# llm = ChatOllama(
#     model="qwen3.6:latest",
#     base_url="http://192.168.178.66:11434",
#     temperature=0,
#     reasoning=False,
#     keep_alive="24h",
#     format="json"
# )

llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("CLAUDE_API_KEY")
)

llm.invoke("Say hello in one sentence.")

AIMessage(content="Hello! I'm happy to help you with whatever you need.", additional_kwargs={}, response_metadata={'id': 'msg_01F6peSANUMKY3qbVaWfqFz9', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 13, 'output_tokens': 16, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019ef459-e955-7373-b270-2d93a394ccb6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 16, 'total_tokens': 29, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}})

In [6]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` 
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session(database="cloud3") as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    print("Index creation failed")
    pass

# Close the driver connection
driver.close()

Index creation failed


In [54]:

class Entities(BaseModel):
    """Identifying information about entities."""

    names: list[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)


entity_chain = prompt | llm.with_structured_output(Entities, method="function_calling")

In [55]:
entity_chain.invoke("Who are Sherlock Holmes and Irene Adler?")

Entities(names=['Sherlock Holmes', 'Irene Adler'])

In [56]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


# Fulltext index query
def graph_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke(question)

    for entity in entities.names:
        response = graph.query(
            """
            CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})
            YIELD node, score
            CALL (node) {
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }
            RETURN output
            LIMIT 50
            """,
            {"query": entity},
        )
        result += "\n".join(el["output"] for el in response) + "\n"

    return result

In [57]:
print(graph_retriever("Who is Irene Adler?"))

Irene Adler - RESIDES_IN -> Briony Lodge
Irene Adler - BORN_IN -> New Jersey
Irene Adler - PERFORMED_AT -> La Scala
Irene Adler - WAS_PRIMA_DONNA_OF -> Imperial Opera of Warsaw
Irene Adler - LIVES_IN -> London
Irene Adler - POSSESSES_COMPROMISING_PHOTOGRAPH_OF -> King of Bohemia
Irene Adler - BLACKMAILS -> King of Bohemia
Irene Adler - THREATENS_TO_EXPOSE -> King of Bohemia
Irene Adler - THREATENS_WITH_PHOTOGRAPH -> King of Bohemia
Irene Adler - RETURNED_TO -> Briony Lodge
Irene Adler - ARRIVED_AT -> Briony Lodge
Irene Adler - RUSHED_TO -> Church of St. Monica
Irene Adler - REWARDED -> Holmes
Irene Adler - RETURNS_TO -> Briony Lodge
Irene Adler - DEPARTED_FROM -> Charing Cross
Irene Adler - MARRIED_TO -> Godfrey Norton
Irene Adler - STORES_PHOTOGRAPH_IN -> Briony Lodge
Irene Adler - ENTERED -> Briony Lodge
Irene Adler - STOOD_AT_ENTRANCE_OF -> Briony Lodge
Irene Adler - INVITED_INTO_HOME -> Holmes
Irene Adler - CARED_FOR -> Holmes
Irene Adler - REVEALED_PHOTOGRAPH_LOCATION_TO -> Holmes

In [ ]:
vector_data = [el.page_content for el in vector_retriever.invoke("Who is Irene Adler?")]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

In [59]:
def full_retriever(question: str):
    graph_data = graph_retriever(question)
    vector_data = [el.page_content for el in vector_retriever.invoke(question)]
    final_data = f"""Graph data:
{graph_data}
vector data:
{"#Document ". join(vector_data)}
    """
    return final_data

In [60]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
        {
            "context": full_retriever,
            "question": RunnablePassthrough(),
        }
    | prompt
    | llm
    | StrOutputParser()
)

In [62]:
chain.invoke(input="How are Irene Adler and Sherlock Holmes related?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

'Sherlock Holmes is an admirer of Irene Adler, who outwitted him and defeated his attempts to retrieve a compromising photograph.'

In [74]:
chain.invoke(input="How is Irene Adler related to the Bohemia?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

'Irene Adler possesses a compromising photograph of the King of Bohemia and blackmails him with it.'

In [12]:
from langchain_core.prompts.prompt import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """Task: Generate Cypher statement to query a graph database.
...
"""
CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
)
CYPHER_QA_TEMPLATE = """You are an assistant that helps to form nice and human understandable answers.
...
"""
CYPHER_QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"], template=CYPHER_QA_TEMPLATE
)
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    qa_prompt=CYPHER_QA_PROMPT,
    verbose=True,
    allow_dangerous_requests=True,
    
)

In [33]:
vector_store = Neo4jVector.from_documents(
    documents,
    OllamaEmbeddings(
        model="mxbai-embed-large",
    ),
)

In [13]:
graph_chain.invoke({"query": "Who is Irene Adler?"})




> Entering new GraphCypherQAChain chain...
Generated Cypher:
Please provide the specific question or requirement you'd like me to translate into a Cypher statement. For example:

- "Find all users who have posted at least one article"
- "Return the names of people who are friends with Alice and Bob"
- "Count the number of nodes labeled Person"

Also, if there's a specific schema (node labels, relationship types, properties), please share it so I can generate an accurate query.


CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'Please': expected 'ALTER', 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FILTER', 'FINISH', 'FOR', 'FOREACH', 'GRANT', 'INSERT', 'LET', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE', 'WHEN', 'WITH' or '{' (line 1, column 1 (offset: 0))
"Please provide the specific question or requirement you'd like me to translate into a Cypher statement. For example:"
 ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

In [4]:
index_name = "vector"
keyword_index_name = "keyword"
store = Neo4jVector.from_existing_index(
    OllamaEmbeddings(
        model="mxbai-embed-large",
    ),
    index_name=index_name,
    keyword_index_name=keyword_index_name,
    search_type='hybrid'
)
graph_chain.run("most used topic in the given dataset?")

NameError: name 'graph_chain' is not defined

In [35]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = graph_chain.run(question)
    unstructured_data = [el.page_content for el in store.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    print(f"Final context: {final_data}")
    return final_data

In [ ]:
template = """Answer the question based only on the following context:
{context}
Question: {query}
"""
suite = ChatPromptTemplate.from_template(template)
retrieval_chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
retrieval_chain.invoke("What are the top themes in this dataset?")
retrieval_chain.invoke("Most discussed clauses?")

In [9]:
class Entity(BaseModel):
    type: str
    name: str

class Output(BaseModel):
    entities: list[Entity]

prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        "You are extracting organization and person entities from the text.",
    ),
    (
        "human",
        "Use the given format to extract information from the following "
        "input: {question}",
    ),
]
)


entity_chain = llm.with_structured_output(Output)

In [14]:
entity_chain.invoke("Who is Sherlock Holmes admiring?").entities

[Entity(type='person', name='Sherlock Holmes'),
 Entity(type='event', name='admiration')]

In [ ]:
from langchain_graph_retriever.document_graph import create_graph, group_by_community


In [18]:
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)
chain.invoke({"query":"Who is Sherlock Holmes?"})




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: 'Sherlock Holmes'})
RETURN p
Full Context:
[{'p': {'id': 'Sherlock Holmes'}}]

> Finished chain.


{'query': 'Who is Sherlock Holmes?', 'result': "I don't know the answer."}

In [112]:
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.llm import AnthropicLLM

from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.embeddings import OpenAIEmbeddings

# 1. Neo4j driver
URI = "neo4j://127.0.0.1:7687"
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))

INDEX_NAME = "ba-cloud-index"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# 2. Retriever
# Create Embedder object, needed to convert the user question (text) to a vector
embedder = OllamaEmbeddings(model="mxbai-embed-large")

# Initialize the retriever
retriever = VectorRetriever(driver, INDEX_NAME, embedder)

# 3. LLM
# Note: the OPENAI_API_KEY must be in the env vars
# llm = AnthropicLLM(
#     model_name="claude-haiku-4-5-20251001",
#     model_params={"max_tokens": 1000},
#     api_key=os.getenv("CLAUDE_API_KEY"),   # can also be read from env vars
# )
llm.invoke("Who is Irene Adler?")

# Initialize the RAG pipeline
rag = GraphRAG(retriever=retriever, llm=llm)

# Query the graph
query_text = "How do I do similarity search in Neo4j?"
response = rag.search(query_text=query_text, retriever_config={"top_k": 5})
print(response.answer)

Exception: No index with name ba-cloud-index found